# AirSense AI Runbook

Run this notebook cell by cell to train and verify the core AirSense AI pipeline on your machine.

This notebook covers preprocessing, model training, saved artifact verification, forecasting, SHAP explainability, and recommendation-agent smoke tests.

After this works locally, you can add the API and frontend on top of the saved artifacts.

## 1) Setup

If needed, install dependencies first from the project root:

```bash
pip install -r AirSense-AI/requirements.txt
```

Then run the notebook with the same Python environment used for the project.

In [1]:
from pathlib import Path
import json
import sys

workspace_root = Path(r'c:/Users/sajal/Downloads/ET AI')
project_root = workspace_root / 'AirSense-AI'
sys.path.insert(0, str(project_root))

print('Project root:', project_root)
print('Saved models folder:', project_root / 'saved_models')

Project root: c:\Users\sajal\Downloads\ET AI\AirSense-AI
Saved models folder: c:\Users\sajal\Downloads\ET AI\AirSense-AI\saved_models


## 2) Load and merge all city CSVs

This confirms the preprocessing layer can find the five CSV files in the workspace root and combine them into one dataframe.

In [2]:
from src.preprocessing import load_all_city_data, fit_preprocessing_artifacts, create_time_series_windows

df = load_all_city_data()
print('Merged rows:', len(df))
print('Cities:', sorted(df['City'].astype(str).unique().tolist()))
print(df.head(3))

Merged rows: 12042
Cities: ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai']
        City      Date   AQI  PM2.5   PM10    NO2    SO2    CO    O3  \
0  Bangalore  01/01/18  68.0   37.4  73.44  56.44  77.52  0.71  64.6   
1  Bangalore  02/01/18  76.0   41.8  82.08  63.08  86.64  0.80  72.2   
2  Bangalore  03/01/18  70.0   38.5  75.60  58.10  79.80  0.74  66.5   

                 source_file  ...  Unnamed: 12  Unnamed: 13  Unnamed: 14  \
0  Bangalore_AQI_Dataset.csv  ...          NaN          NaN          NaN   
1  Bangalore_AQI_Dataset.csv  ...          NaN          NaN          NaN   
2  Bangalore_AQI_Dataset.csv  ...          NaN          NaN          NaN   

   Unnamed: 15  Unnamed: 16  Unnamed: 17  Unnamed: 18  Unnamed: 19  \
0          NaN          NaN          NaN          NaN          NaN   
1          NaN          NaN          NaN          NaN          NaN   
2          NaN          NaN          NaN          NaN          NaN   

   Unnamed: 20  Unnamed: 21  
0          

## 3) Preprocessing smoke test

This checks that artifacts can be fitted and that time-series windows are created correctly.

In [3]:
artifacts = fit_preprocessing_artifacts(df, window_size=14)
X, y, dates, cities = create_time_series_windows(df, artifacts, window_size=14)

print('Window shape:', X.shape)
print('Target shape:', y.shape)
print('Unique cities in windows:', sorted(set(cities)))

c:\Users\sajal\Downloads\ET AI\AirSense-AI\src\preprocessing.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cleaned["Date"] = pd.to_datetime(cleaned["Date"], dayfirst=True, errors="coerce")
c:\Users\sajal\Downloads\ET AI\AirSense-AI\src\preprocessing.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cleaned["Date"] = pd.to_datetime(cleaned["Date"], dayfirst=True, errors="coerce")


Window shape: (11972, 14, 12)
Target shape: (11972,)
Unique cities in windows: ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai']


## 4) Train all models

This trains the baseline Random Forest, XGBoost if available, the LSTM, and the Transformer.

If you only want a fast smoke test, keep the epoch counts small. Increase them later for better results.

In [4]:
from src.models.train import train_all_models

result = train_all_models(
    output_dir=project_root / 'saved_models',
    lstm_epochs=1,
    transformer_epochs=1,
)

print(json.dumps(result['summary'], indent=2, default=str))

c:\Users\sajal\Downloads\ET AI\AirSense-AI\src\preprocessing.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cleaned["Date"] = pd.to_datetime(cleaned["Date"], dayfirst=True, errors="coerce")
c:\Users\sajal\Downloads\ET AI\AirSense-AI\src\preprocessing.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cleaned["Date"] = pd.to_datetime(cleaned["Date"], dayfirst=True, errors="coerce")


{
  "window_size": 14,
  "feature_names": [
    "lag_14_AQI",
    "lag_14_PM2.5",
    "lag_14_PM10",
    "lag_14_NO2",
    "lag_14_SO2",
    "lag_14_CO",
    "lag_14_O3",
    "lag_14_city_encoded",
    "lag_14_day_of_week",
    "lag_14_month",
    "lag_14_day_of_year",
    "lag_14_is_weekend",
    "lag_13_AQI",
    "lag_13_PM2.5",
    "lag_13_PM10",
    "lag_13_NO2",
    "lag_13_SO2",
    "lag_13_CO",
    "lag_13_O3",
    "lag_13_city_encoded",
    "lag_13_day_of_week",
    "lag_13_month",
    "lag_13_day_of_year",
    "lag_13_is_weekend",
    "lag_12_AQI",
    "lag_12_PM2.5",
    "lag_12_PM10",
    "lag_12_NO2",
    "lag_12_SO2",
    "lag_12_CO",
    "lag_12_O3",
    "lag_12_city_encoded",
    "lag_12_day_of_week",
    "lag_12_month",
    "lag_12_day_of_year",
    "lag_12_is_weekend",
    "lag_11_AQI",
    "lag_11_PM2.5",
    "lag_11_PM10",
    "lag_11_NO2",
    "lag_11_SO2",
    "lag_11_CO",
    "lag_11_O3",
    "lag_11_city_encoded",
    "lag_11_day_of_week",
    "lag_11_month",
   

## 5) Verify saved artifacts

This confirms the model files were written into `saved_models/`.

In [5]:
saved_models_dir = project_root / 'saved_models'
print(sorted([path.name for path in saved_models_dir.iterdir()]))

['lstm.pt', 'preprocessing.joblib', 'random_forest.joblib', 'training_metrics.json', 'transformer.pt', 'xgboost.joblib']


## 6) Load saved models and forecast a city

Use this to test the inference path after training.

In [6]:
from src.models.predict import load_artifacts, recursive_forecast

loaded = load_artifacts(saved_models_dir)
forecast, _ = recursive_forecast(df, loaded, city='Delhi', model_name='transformer', horizon=5)
forecast

c:\Users\sajal\Downloads\ET AI\AirSense-AI\src\preprocessing.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cleaned["Date"] = pd.to_datetime(cleaned["Date"], dayfirst=True, errors="coerce")


[{'date': datetime.datetime(2025, 1, 1, 0, 0),
  'predicted_aqi': 316.15645101775243},
 {'date': datetime.datetime(2025, 1, 2, 0, 0),
  'predicted_aqi': 323.43311052131907},
 {'date': datetime.datetime(2025, 1, 3, 0, 0),
  'predicted_aqi': 314.28849538837494},
 {'date': datetime.datetime(2025, 1, 4, 0, 0),
  'predicted_aqi': 303.9874083769127},
 {'date': datetime.datetime(2025, 1, 5, 0, 0),
  'predicted_aqi': 297.6094181196612}]

## 7) SHAP explainability smoke test

This returns pollutant contribution percentages for one city.

In [9]:
from src.preprocessing import latest_sequence_for_city, build_feature_names
from src.explainability.shap_analysis import explain_tree_model

sequence_window, _ = latest_sequence_for_city(df, loaded.artifacts, 'Delhi')
feature_names = build_feature_names(loaded.artifacts.window_size, loaded.artifacts.feature_columns)
tree_model = loaded.random_forest or loaded.xgboost
explanation = explain_tree_model(tree_model, sequence_window[None, ...].reshape(1, -1), feature_names)
explanation.pollutant_contributions[:5]

c:\Users\sajal\Downloads\ET AI\AirSense-AI\src\preprocessing.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cleaned["Date"] = pd.to_datetime(cleaned["Date"], dayfirst=True, errors="coerce")


[{'feature': 'PM10', 'importance': 18.81},
 {'feature': 'PM2.5', 'importance': 18.54},
 {'feature': 'SO2', 'importance': 17.0},
 {'feature': 'CO', 'importance': 16.59},
 {'feature': 'O3', 'importance': 14.87}]

## 8) Recommendation agent smoke test

This converts the forecast into risk level and intervention suggestions.

In [10]:
from src.agents.recommendation_agent import generate_recommendations

predicted_aqi = forecast[0]['predicted_aqi']
dominant_pollutants = [(item['feature'], item['importance']) for item in explanation.pollutant_contributions[:3]]
plan = generate_recommendations('Delhi', predicted_aqi, dominant_pollutants)
plan

InterventionPlan(city='Delhi', predicted_aqi=316.15645101775243, category='Very Poor', risk_level='Very High', interventions=['Reduce construction activity in exposed zones.', 'Restrict heavy vehicle movement during peak hours.', 'Increase roadside and industrial pollution monitoring.', 'PM10 (18.8%) - Control road dust, deploy vacuum sweeping, and enforce site-level particulate barriers.', 'PM2.5 (18.5%) - Strengthen dust suppression, construction-site sealing, and public transport prioritization.', 'SO2 (17.0%) - Audit industrial emissions and increase stack monitoring in affected zones.'], health_advisory='Avoid outdoor exposure, especially for sensitive groups.')

## What to do next

1. Run the notebook from top to bottom.
2. If training succeeds, the artifacts will appear in `saved_models/`.
3. Only after that, build the FastAPI backend and React frontend on top of these saved artifacts.